# Counterfactual evaluation

This notebook evaluates image replacement (random, same-label, opposite-label), image ablation, and controlled text perturbation. Ground-truth labels are never changed after an intervention. Set `CHECKPOINT_PATH` to an executed research checkpoint.

In [1]:
%pip -q install transformers accelerate scikit-learn pyyaml tqdm matplotlib seaborn


In [2]:
import os, sys
from pathlib import Path
import torch

SMOKE_TEST = True
LIMIT = 16 if SMOKE_TEST else None
REPO_ROOT_OVERRIDE = ""
CHECKPOINT_PATH = ""  # e.g. /kaggle/working/research_outputs/cross_attention_vit/best_model.pt
CONFIG_PATH = ""      # matching YAML; required when CHECKPOINT_PATH is set
candidates = [Path(REPO_ROOT_OVERRIDE)] if REPO_ROOT_OVERRIDE else []
candidates += [Path("/kaggle/working/FakeNewsDetector"), Path("/content/FakeNewsDetector"), Path.cwd()]
REPO_ROOT = next((p for p in candidates if (p / "src/experiments/run_experiment.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT_OVERRIDE to the uploaded repository root.")
os.environ["PYTHONPATH"] = str(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
OUTPUT_ROOT = Path("/kaggle/working/research_outputs" if Path("/kaggle").exists() else "/content/research_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.is_available())

FileNotFoundError: Set REPO_ROOT_OVERRIDE to the uploaded repository root.

In [ ]:
import pandas as pd
from PIL import Image
from src.experiments.counterfactual import build_image_interventions, perturb_text, load_rgb, summarize_records

frame = pd.read_csv(REPO_ROOT / "data/splits/test.csv")
samples = build_image_interventions(frame, str(REPO_ROOT / "data/raw/images"), limit=LIMIT)
for sample in samples:
    sample["images"] = {key: (load_rgb(value) if value else None) for key, value in sample["images"].items()}
    sample["images"]["ablated"] = None
    sample["perturbed_text"] = perturb_text(sample["text"], "mask_claim")
print("Intervention samples:", len(samples))
print("Conditions:", list(samples[0]["images"]))

In [ ]:
if not CHECKPOINT_PATH or not CONFIG_PATH:
    print("Set CHECKPOINT_PATH and CONFIG_PATH, then rerun this cell to score interventions.")
else:
    import yaml
    import numpy as np
    from transformers import RobertaTokenizer, ViTImageProcessor
    from src.models.research_model import ResearchMultimodalModel
    from src.models.proposed.counterfactual import evaluate_counterfactuals

    with open(CONFIG_PATH, encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = RobertaTokenizer.from_pretrained(config["model"]["text_encoder"]["checkpoint"])
    visual = config["model"].get("visual_encoder", {})
    processor = ViTImageProcessor.from_pretrained(visual.get("checkpoint", "google/vit-base-patch16-224"))
    model = ResearchMultimodalModel(config).to(device)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    def predict(text, image):
        if image is None:
            image = Image.new("RGB", (224, 224), color="black")
        tokens = tokenizer(text, max_length=config["dataset"]["max_seq_length"], padding="max_length", truncation=True, return_tensors="pt")
        pixels = processor(images=image, return_tensors="pt")
        tokens = {key: value.to(device) for key, value in tokens.items()}
        pixels = {key: value.to(device) for key, value in pixels.items()}
        with torch.no_grad():
            probabilities = torch.softmax(model(tokens, pixels, [text]), dim=-1)[0]
        label = int(probabilities.argmax())
        return label, float(probabilities[label])

    cf_samples = [{"id": s["id"], "text": s["text"], "label": s["label"], "images": s["images"]} for s in samples]
    records = evaluate_counterfactuals(cf_samples, predict)
    summary = summarize_records(records)
    text_rows = []
    for sample in samples:
        original_label, original_confidence = predict(sample["text"], sample["images"]["original"])
        changed_text = perturb_text(sample["text"], "mask_claim")
        changed_label, changed_confidence = predict(changed_text, sample["images"]["original"])
        text_rows.append({"id": sample["id"], "condition": "text_mask_claim", "true_label": sample["label"], "predicted_label": changed_label, "prediction_changed": changed_label != original_label, "confidence_delta": changed_confidence - original_confidence})
    text_summary = pd.DataFrame(text_rows)
    summary.to_csv(OUTPUT_ROOT / "counterfactual_summary.csv", index=False)
    text_summary.to_csv(OUTPUT_ROOT / "text_perturbation_records.csv", index=False)
    display(summary)
    display(text_summary.describe(include="all"))

The replacement conditions measure modality sensitivity. Opposite-label images are metadata-controlled interventions, not relabeled examples. Text perturbation uses the same principle and should be evaluated with a text/image prediction callback when a checkpoint is configured.

# Visual outputs

These figures show prediction-change rates and confidence changes under image and text interventions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
summary_path = OUTPUT_ROOT / "counterfactual_summary.csv"
text_path = OUTPUT_ROOT / "text_perturbation_records.csv"
if summary_path.exists():
    counterfactual = pd.read_csv(summary_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=counterfactual, x="condition", y="prediction_change_rate", ax=axes[0], color="#4c78a8")
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Prediction change rate")
    axes[0].tick_params(axis="x", rotation=25)
    sns.barplot(data=counterfactual, x="condition", y="mean_confidence_delta", ax=axes[1], color="#f58518")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Mean confidence change")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "counterfactual_sensitivity.png", dpi=160)
    plt.show()
else:
    print("No counterfactual summary found; configure a checkpoint and rerun the scoring cell.")

if text_path.exists():
    text_results = pd.read_csv(text_path)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=text_results, x="condition", y="confidence_delta", ax=ax, color="#54a24b")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Text perturbation confidence change")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "text_perturbation_confidence.png", dpi=160)
    plt.show()